# Fundamentals 05 - Lineage Memory API

Objetivo: derivar memoria y evidencia desde `RunResult` reales, sin fabricar pasos ni mutar el resultado final.

## Parametros de la demostracion

| Parametro | Default | Proposito |
|---|---|---|
| symbols | tool y graph | Generar dos RunResult reales relacionados. |
| lineage | toolkit.lineage | Conservar procedencia sin mutar resultados. |
| composition | toolkit.compose_result | Crear el resultado agregado desde evidencia real. |

## 1) Ejecutar dos observaciones

In [ ]:
import agentic_systems as toolkit

runtime = toolkit.runtime(provider="python-runtime")
system = toolkit.system(runtime=runtime)

@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {
        "symbol": symbol,
        "is_public": symbol in toolkit.PUBLIC_API,
        "package_version": toolkit.__version__,
    }

agent = system.agent(
    name="public_api_inspector",
    instructions="Ejecuta inspect_public_api y conserva la evidencia observada.",
    tools=[inspect_public_api],
    runtime=runtime,
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
    policy=toolkit.RunPolicy(max_tool_calls=1, max_turns=2, temperature=0.0),
)

first_result = agent.run(
    {"tool": "inspect_public_api", "input": {"symbol": "LineageMemory"}}, mode="eval"
)
second_result = agent.run(
    {"tool": "inspect_public_api", "input": {"symbol": "environment"}}, mode="eval"
)
assert first_result.ok and second_result.ok

## 2) Proyectar Lineage desde resultados

`RunResult.lineage` usa tool events, validacion y runtime ya observados.

In [ ]:
first_lineage = first_result.lineage(
    name="tutorial.lineage.first",
    question="LineageMemory es parte de la API publica?",
    goal="Explicar evidencia de una ejecucion.",
)
toolkit.show(first_lineage, title="First Lineage")
toolkit.show_json({
    "prompt_context": first_lineage.to_prompt_context(max_chars=900),
}, title="Compact prompt context")

## 3) Componer ejecuciones sin mutarlas

`compose_result` agrega metadata desde ambos resultados reales. No se asigna `result.final` ni se inventa runtime.

In [ ]:
combined = toolkit.compose_result(
    text="Se inspeccionaron dos simbolos de la API publica.",
    data={
        "observations": [
            toolkit.agent_output(first_result),
            toolkit.agent_output(second_result),
        ]
    },
    results=[first_result, second_result],
    mode="lineage",
    input={"symbols": ["LineageMemory", "environment"]},
)

toolkit.human_result(combined, title="Composed RunResult", show_lineage=True)
combined_lineage = combined.lineage(
    name="tutorial.lineage.combined",
    question="Que evidencia produjeron las dos ejecuciones?",
    goal="Conservar lineage compuesto sin mutaciones.",
)
toolkit.show(combined_lineage, title="Combined Lineage")

## 4) API realmente ejercitada

In [ ]:
api_coverage = [
    "agent.run", "RunResult.lineage", "LineageMemory.to_prompt_context",
    "toolkit.agent_output", "toolkit.compose_result", "toolkit.human_result",
    "toolkit.show", "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="Lineage API coverage")

## Resultado esperado

Dos ejecuciones de origen, un resultado compuesto y lineage trazable hasta las Tools reales.